In [3]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

import gradio as gr
from datetime import datetime
from pathlib import Path
import json
import re
import ast
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle, Preformatted
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
import requests
import urllib.parse
import inspect

In [5]:
class DocGenieAnalyzer:
    """Analyze Python functions and generate professional docstrings"""

    @staticmethod
    def extract_function_signature(code):
        try:
            tree = ast.parse(code)
            func_def = tree.body[0]

            if not isinstance(func_def, ast.FunctionDef):
                return None, None

            func_name = func_def.name
            params = []

            for arg in func_def.args.args:
                param_type = "Any"
                if arg.annotation and hasattr(ast, "unparse"):
                    param_type = ast.unparse(arg.annotation)

                params.append({
                    "name": arg.arg,
                    "type": param_type,
                    "default": None
                })

            for i, default in enumerate(func_def.args.defaults):
                idx = len(params) - len(func_def.args.defaults) + i
                if idx >= 0:
                    params[idx]["default"] = str(default)

            return_type = "Any"
            if func_def.returns and hasattr(ast, "unparse"):
                return_type = ast.unparse(func_def.returns)

            return {
                "name": func_name,
                "params": params,
                "return_type": return_type
            }, func_def

        except:
            return None, None


    @staticmethod
    def analyze_function_logic(func_def, code):

        try:
            analysis = {
                "has_loops": False,
                "has_conditions": False,
                "has_return": False,
                "operations": [],
                "description": ""
            }

            for node in ast.walk(func_def):

                if isinstance(node, (ast.For, ast.While)):
                    analysis["has_loops"] = True

                elif isinstance(node, ast.If):
                    analysis["has_conditions"] = True

                elif isinstance(node, ast.Return):
                    analysis["has_return"] = True

                elif isinstance(node, ast.BinOp):

                    if isinstance(node.op, ast.Mult):
                        analysis["operations"].append("multiplication")

                    elif isinstance(node.op, ast.Add):
                        analysis["operations"].append("addition")

                    elif isinstance(node.op, ast.Sub):
                        analysis["operations"].append("subtraction")

                    elif isinstance(node.op, ast.Div):
                        analysis["operations"].append("division")

            desc = []

            desc.append(f"Executes {func_def.name} operation")

            if analysis["has_conditions"]:
                desc.append("with conditional logic")

            if analysis["has_loops"]:
                desc.append("and loops through values")

            if analysis["operations"]:
                desc.append("performing calculations")

            if analysis["has_return"]:
                desc.append("and returns the result")

            analysis["description"] = " ".join(desc) + "."

            return analysis

        except:
            return {"description": "Executes function logic."}


    @staticmethod
    def generate_google_docstring(signature, analysis):

        summary = analysis.get("description")

        docstring = f'    """{summary}\n'

        if signature["params"]:
            docstring += "\n    Args:\n"

            for param in signature["params"]:

                default = ""
                if param["default"]:
                    default = f"(default={param['default']})"

                docstring += f"        {param['name']} ({param['type']}): parameter {default}\n"

        docstring += f"\n    Returns:\n        {signature['return_type']}: result value\n"
        docstring += '    """\n'

        return docstring


    @staticmethod
    def generate_numpy_docstring(signature, analysis):

        summary = analysis.get("description")

        docstring = f'    """{summary}\n'

        if signature["params"]:
            docstring += "\n    Parameters\n    ----------\n"

            for param in signature["params"]:

                default = ""
                if param["default"]:
                    default = f", default={param['default']}"

                docstring += f"    {param['name']} : {param['type']}{default}\n"
                docstring += "        Input parameter\n"

        docstring += "\n    Returns\n    -------\n"
        docstring += f"    {signature['return_type']}\n"
        docstring += "        Result value\n"

        docstring += '    """\n'

        return docstring


analyzer = DocGenieAnalyzer()
generation_history = []

In [7]:
def generate_docstring(code, style):

    if not code.strip():
        return "", "❌ Empty code"

    signature, func_def = analyzer.extract_function_signature(code)

    if not signature:
        return "", "❌ Invalid function"

    analysis = analyzer.analyze_function_logic(func_def, code)

    if style == "google":
        docstring = analyzer.generate_google_docstring(signature, analysis)

    else:
        docstring = analyzer.generate_numpy_docstring(signature, analysis)

    lines = code.split("\n")

    final_code = lines[0] + "\n" + docstring + "\n".join(lines[1:])

    generation_history.append({
        "code": code,
        "docstring": docstring,
        "style": style,
        "time": datetime.now().isoformat()
    })

    return final_code, "✅ Docstring generated"

In [8]:
def process_upload(file):

    if not file:
        return "", "❌ No file uploaded"

    try:
        content = Path(file.name).read_text()
        return content, "✅ File loaded"

    except:
        return "", "❌ Failed to read file"

In [9]:
def download_pdf():

    if not generation_history:
        return "❌ No data", None

    filename = "doc_genie_report.pdf"

    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("Doc-Genie Documentation", styles["Title"]))
    story.append(Spacer(1,20))

    for item in generation_history:

        story.append(Paragraph("Original Code", styles["Heading2"]))
        story.append(Preformatted(item["code"], styles["Code"]))

        story.append(Paragraph("Generated Docstring", styles["Heading2"]))
        story.append(Preformatted(item["docstring"], styles["Code"]))

        story.append(Spacer(1,20))

    doc = SimpleDocTemplate(filename, pagesize=A4)
    doc.build(story)

    return "✅ PDF ready", filename


In [10]:
def generate_whatsapp_link():

    if not generation_history:
        return "No data"

    text = "Doc-Genie generated documentation"

    encoded = urllib.parse.quote(text)

    return f"https://wa.me/?text={encoded}"


def generate_facebook_link():

    text = "Generated Python documentation using Doc-Genie"

    encoded = urllib.parse.quote(text)

    return f"https://www.facebook.com/sharer/sharer.php?quote={encoded}"

In [11]:
with gr.Blocks(title="Doc-Genie") as demo:

    gr.Markdown("# 📚 Doc-Genie\nPython Docstring Generator")

    with gr.Row():

        with gr.Column(scale=3):

            code_input = gr.Code(
                language="python",
                lines=10,
                value="def add(a,b):\n    return a+b"
            )

            file_upload = gr.File(label="Upload .py file")
            upload_status = gr.Textbox()

            file_upload.change(
                process_upload,
                inputs=file_upload,
                outputs=[code_input, upload_status]
            )

            style = gr.Radio(["google","numpy"], value="google")

            generate_btn = gr.Button("Generate")

            output_code = gr.Code(language="python", lines=15)

            status = gr.Textbox()

            generate_btn.click(
                generate_docstring,
                inputs=[code_input, style],
                outputs=[output_code, status]
            )

        with gr.Column(scale=1):

            pdf_btn = gr.Button("Download PDF")

            pdf_file = gr.File()

            pdf_status = gr.Textbox()

            pdf_btn.click(
                download_pdf,
                outputs=[pdf_status, pdf_file]
            )

            whatsapp_btn = gr.Button("WhatsApp Share")
            facebook_btn = gr.Button("Facebook Share")

            share_box = gr.Textbox()

            whatsapp_btn.click(generate_whatsapp_link, outputs=share_box)
            facebook_btn.click(generate_facebook_link, outputs=share_box)


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5bf6e85d715d024412.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
